### IDENTIFYING PATHWAYS FOR DEEPER FINANCIAL ENGAGEMENT FOR RMG WORKERS: BASELINE EVALUATION DRAFT QUESTIONNAIRE

### Setting up the working directories, importing data, filtering data and preparing code for the cleaning

In [57]:
# libraries
import os
import pandas as pd
import pyreadstat

#### Setting paths

In [58]:
# setting workind diectory for data import
data_path = "D:/OneDrive/Dalberg Research Projects/BRAC/Data"
os.chdir(data_path)
print(f"Directory set to: {os.getcwd()}")

Directory set to: D:\OneDrive\Dalberg Research Projects\BRAC\Data


In [59]:
# Setting the output path and create folder if it doesn't exist
from datetime import datetime

# define output path
output_path = "D:/OneDrive/Dalberg Research Projects/BRAC/Error Logs"
# get the current date in YYYY-MM-DD format
today_str = datetime.now().strftime("%Y-%m-%d")
daily_output_path = os.path.join(output_path, today_str)

# 4. Create the folder if it doesn't exist
if not os.path.exists(daily_output_path):
    os.makedirs(daily_output_path)
    print(f"Created new folder: {daily_output_path}")
else:
    print(f"Folder already exists: {daily_output_path}")

# Initialize master error log dictionary
master_error_log = {}

Folder already exists: D:/OneDrive/Dalberg Research Projects/BRAC/Error Logs\2025-12-22


In [60]:
# load the SPSS file
file_name = "BRAC - BASELINE EVALUATION_WIDE V006.sav"

# Load data AND metadata
df, meta = pyreadstat.read_sav(file_name, apply_value_formats=False)
label_dict = meta.column_names_to_labels

# Define the new labels for the C2c variables
new_labels = {
    "C2c_1": "C2c. How often are you Applying for a loan in the Agami App?",
    "C2c_2": "C2c. How often are you Checking loan balance in the Agami App?",
    "C2c_3": "C2c. How often are you Checking loan repayment date & installment amount in the Agami App?",
    "C2c_4": "C2c. How often are you Using loan calculator in the Agami App?",
    "C2c_5": "C2c. How often are you Checking loan products in the Agami App?",
    "C2c_6": "C2c. How often are you Checking savings balance in the Agami App?",
    "C2c_7": "C2c. How often are you Checking savings reminder in the Agami App?",
    "C2c_8": "C2c. How often are you Using savings calculator in the Agami App?",
    "C2c_9": "C2c. How often are you Checking savings products in the Agami App?",
    "C2c_10": "C2c. How often are you Viewing app offers in the Agami App?",
    "C2c_11": "C2c. How often are you Checking insurance products in the Agami App?",
    "C2c_12": "C2c. How often are you Seeking support through 'contact' option in the Agami App?",
    "C2c_13": "C2c. How often are you Checking or verifying my transactions independently in the Agami App?"
}

# Update your existing label dictionary
label_dict.update(new_labels)

# Verification: Print one to confirm the change
print("New Label for C2c_1:", label_dict["C2c_1"])

# drop unnecessary columns
variables_to_drop = ["SubmissionDate", "starttime", "endtime", "deviceid", "devicephonenum", "username", "device_info", "caseid", "password", "instanceID", "formdef_version"]
df.drop(columns=variables_to_drop, inplace=True, errors='ignore')
len(df)

New Label for C2c_1: C2c. How often are you Applying for a loan in the Agami App?


444

#### Filter data to keep only necessary cases

In [61]:
# old data file
old_data = "BRAC - BASELINE EVALUATION_WIDE V005.sav"
id_col = "KEY"

if os.path.exists(old_data):
    # Load previously processed IDs from SPSS
    old_ids_df = pd.read_spss(old_data, usecols=[id_col], convert_categoricals=False)
    processed_ids = set(old_ids_df[id_col])
    print(f"Loaded {len(processed_ids)} previously processed IDs from SPSS.")
else:
    processed_ids = set()
    print("No history file found. Treating all data as new.")

# filter for new data only
df[id_col] = df[id_col].astype(str)
data_new = df[~df[id_col].isin(processed_ids)].copy()

print(f"Total rows in daily file: {len(df)}")
print(f"New rows to process: {len(data_new)}")

df = data_new
len(df)

Loaded 304 previously processed IDs from SPSS.
Total rows in daily file: 444
New rows to process: 140


140

#### Data Screening

In [62]:
# Was the first call successful?
q1_mapping = meta.variable_value_labels.get("Q_1", {})
counts = df['Q_1'].map(q1_mapping).fillna(df['Q_1']).value_counts()
q1_table = pd.DataFrame({
    'Count': counts,
    'Percentage': (counts / counts.sum() * 100).round(1)
})
q1_table.loc['Total'] = q1_table.sum()

print("Number of succesful first calls (Q_1):")
print(q1_table)

Number of succesful first calls (Q_1):
       Count  Percentage
Q_1                     
Yes    140.0       100.0
Total  140.0       100.0


In [63]:
# Q2.1.	Based on your available schedule, a male enumerator may come and take your interview. Would you be comfortable being interviewed by a male enumerator?
Q2_mapping = meta.variable_value_labels.get("Q2_1", {})
counts = df['Q2_1'].map(Q2_mapping).fillna(df['Q2_1']).value_counts()
q1_table = pd.DataFrame({
    'Count': counts,
    'Percentage': (counts / counts.sum() * 100).round(1)
})
q1_table.loc['Total'] = q1_table.sum()

print("Respondent preference of interviewer gender:")
print(q1_table)

Respondent preference of interviewer gender:
                                    Count  Percentage
Q2_1                                                 
No, I am comfortable                 71.0        50.7
Yes, I prefer a female interviewer   69.0        49.3
Total                               140.0       100.0


In [64]:
# B5. How easy or difficult was it to log in the first time? 
B5_mapping = meta.variable_value_labels.get("B5", {})
counts = df['B5'].map(B5_mapping).fillna(df['B5']).value_counts()
B5_table = pd.DataFrame({
    'Count': counts,
    'Percentage': (counts / counts.sum() * 100).round(1)
})
B5_table.loc['Total'] = B5_table.sum()

print("How easy or difficult was it to log in the first time? ")
print(B5_table)

How easy or difficult was it to log in the first time? 
                    Count  Percentage
B5                                   
Very easy            13.0        32.5
Somewhat easy        12.0        30.0
Neutral               8.0        20.0
Somewhat difficult    6.0        15.0
Very difficult        1.0         2.5
Total                40.0       100.0


In [65]:
# B6. Which language do you prefer to use in the app?
B6_mapping = meta.variable_value_labels.get("B6", {})
counts = df['B6'].map(B6_mapping).fillna(df['B6']).value_counts()
B6_table = pd.DataFrame({
    'Count': counts,
    'Percentage': (counts / counts.sum() * 100).round(1)
})
B6_table.loc['Total'] = B6_table.sum()

print("Which language do you prefer to use in the app? (Valid Cases Only)")
print(B6_table)

Which language do you prefer to use in the app? (Valid Cases Only)
         Count  Percentage
B6                        
Bengali   36.0        90.0
English    4.0        10.0
Total     40.0       100.0


In [66]:
# B7. How easy is it to find what you need in the app?
B7_mapping = meta.variable_value_labels.get("B7", {})
counts = df['B7'].map(B7_mapping).fillna(df['B7']).value_counts()
B7_table = pd.DataFrame({   
    'Count': counts,
    'Percentage': (counts / counts.sum() * 100).round(1)
})
B7_table.loc['Total'] = B7_table.sum()
print("How easy is it to find what you need in the app?")
print(B7_table)

How easy is it to find what you need in the app?
                    Count  Percentage
B7                                   
Very easy            20.0        50.0
Somewhat easy        10.0        25.0
Neutral               7.0        17.5
Somewhat difficult    2.0         5.0
Very difficult        1.0         2.5
Total                40.0       100.0


In [67]:
# B8. Have you ever faced any obstacles/limitations/barriers that make it hard for you to use Agami App regularly?
B8_mapping = meta.variable_value_labels.get("B8", {})
counts = df['B8'].map(B8_mapping).fillna(df['B8']).value_counts()
B8_table = pd.DataFrame({
        'Count': counts,
        'Percentage': (counts / counts.sum() * 100).round(1)
})
B8_table.loc['Total'] = B8_table.sum()
print("Have you ever faced any obstacles/limitations/barriers that make it hard for you to use Agami App regularly?")
print(B8_table)

Have you ever faced any obstacles/limitations/barriers that make it hard for you to use Agami App regularly?
       Count  Percentage
B8                      
No      40.0       100.0
Total   40.0       100.0


In [68]:
# B9. Before joining Agami App, were you using any other digital financial services (DFS)?
B9_mapping = meta.variable_value_labels.get("B9", {})
counts = df['B9'].map(B9_mapping).fillna(df['B9']).value_counts()
B9_table = pd.DataFrame({
        'Count': counts,
        'Percentage': (counts / counts.sum() * 100).round(1)
})
B9_table.loc['Total'] = B9_table.sum()
print("Have you ever faced any obstacles/limitations/barriers that make it hard for you to use Agami App regularly?")
print(B9_table)

Have you ever faced any obstacles/limitations/barriers that make it hard for you to use Agami App regularly?
       Count  Percentage
B9                      
No      78.0        55.7
Yes     62.0        44.3
Total  140.0       100.0


In [69]:
# B10. Do you currently use any other digital financial platforms besides Agami App?
B10_mapping = meta.variable_value_labels.get("B10", {})
counts = df['B10'].map(B10_mapping).fillna(df['B10']).value_counts()
B10_table = pd.DataFrame({
        'Count': counts,
        'Percentage': (counts / counts.sum() * 100).round(1)
})
B10_table.loc['Total'] = B10_table.sum()
print("Have you ever faced any obstacles/limitations/barriers that make it hard for you to use Agami App regularly?")
print(B10_table)

Have you ever faced any obstacles/limitations/barriers that make it hard for you to use Agami App regularly?
       Count  Percentage
B10                     
Yes     71.0        50.7
No      69.0        49.3
Total  140.0       100.0


In [70]:
# C8. Overall, how satisfied are you with using the Agami App? 
C8_mapping = meta.variable_value_labels.get("C8", {})
counts = df['C8'].map(C8_mapping).fillna(df['C8']).value_counts()
C8_table = pd.DataFrame({
        'Count': counts,
        'Percentage': (counts / counts.sum() * 100).round(1)
})
C8_table.loc['Total'] = C8_table.sum()
print("Have you ever faced any obstacles/limitations/barriers that make it hard for you to use Agami App regularly?")
print(C8_table)

Have you ever faced any obstacles/limitations/barriers that make it hard for you to use Agami App regularly?
                    Count  Percentage
C8                                   
Very satisfied       31.0        77.5
Somewhat satisfied    6.0        15.0
Neutral               3.0         7.5
Total                40.0       100.0


In [71]:
# C9. Compared to other digital financial apps or services, how convenient is Agami App for you?
C9_mapping = meta.variable_value_labels.get("C9", {})
counts = df['C9'].map(C9_mapping).fillna(df['C9']).value_counts()
C9_table = pd.DataFrame({
        'Count': counts,
        'Percentage': (counts / counts.sum() * 100).round(1)
})
C9_table.loc['Total'] = C9_table.sum()
print("Have you ever faced any obstacles/limitations/barriers that make it hard for you to use Agami App regularly?")
print(C9_table)

Have you ever faced any obstacles/limitations/barriers that make it hard for you to use Agami App regularly?
                          Count  Percentage
C9                                         
Much more convenient       21.0        52.5
Slightly more convenient   12.0        30.0
About the same              7.0        17.5
Total                      40.0       100.0


In [72]:
# C10. Have you ever been discouraged by anyone from using Agami App or other digital financial services?
C10_mapping = meta.variable_value_labels.get("C10", {})
counts = df['C10'].map(C10_mapping).fillna(df['C10']).value_counts()
C10_table = pd.DataFrame({
        'Count': counts,
        'Percentage': (counts / counts.sum() * 100).round(1)
})
C10_table.loc['Total'] = C10_table.sum()
print("Have you ever faced any obstacles/limitations/barriers that make it hard for you to use Agami App regularly?")
print(C10_table)

Have you ever faced any obstacles/limitations/barriers that make it hard for you to use Agami App regularly?
       Count  Percentage
C10                     
No      40.0       100.0
Total   40.0       100.0


In [73]:
# Check both at once
var_name = 'B3a'
question_text = meta.column_names_to_labels.get(var_name, "No Label Found")
answer_choices = meta.variable_value_labels.get(var_name, {})

print(f"Variable: {question_text}")
print("Answer Choices:", answer_choices)

Variable: B3a. How often do you use the Agami App? 

<span style= "color:blue">[Read out]</span>
Answer Choices: {1.0: 'Daily', 2.0: '2 –3 times a week', 3.0: 'Weekly', 4.0: '2-3 times a month', 5.0: 'Monthly', 6.0: 'Rarely (less than once a month)', 7.0: 'Never'}


In [74]:
var_name = 'C2c_13'
question_text = meta.column_names_to_labels.get(var_name, "No Label Found")
answer_choices = meta.variable_value_labels.get(var_name, {})

print(f"Variable: {question_text}")
print("Answer Choices:", answer_choices)

Variable: C2c. How often are you Checking or verifying my transactions independently in the Agami App?
Answer Choices: {1.0: 'Daily', 2.0: 'Weekly', 3.0: 'Monthly', 4.0: 'Occasionally (1-2 times a quarter)', 5.0: 'Rarely (less than once in a quarter)'}


#### Prepare code book and variable labels

In [75]:
# Python Export Function
def save_error_log(data, label_dict, file_path, sheet_name="Errors"):
    header_labels = [label_dict.get(col, col) for col in data.columns]
    variable_names = data.columns.tolist()

    with pd.ExcelWriter(file_path, engine='xlsxwriter') as writer:
        data.to_excel(writer, sheet_name=sheet_name, startrow=2, header=False, index=False)
        workbook = writer.book
        worksheet = writer.sheets[sheet_name]
        label_format = workbook.add_format({
            'bold': True, 
            'text_wrap': True, 
            'valign': 'top'
        })
        var_name_format = workbook.add_format({
            'bold': True
        })
        for col_num, value in enumerate(header_labels):
            worksheet.write(0, col_num, value, label_format)
            
        for col_num, value in enumerate(variable_names):
            worksheet.write(1, col_num, value, var_name_format)
            
    print(f"Saved: {file_path}")

In [76]:
# variables to keep
variables_to_keep = ["KEY", "INT_DATE", "END_TIME","Pull_Member_Name", "Pull_Contact_Number", "Pull_Region_name", "Pull_Area_Name"] 

### Section A: Demographics 

In [77]:
min_seconds = 15 * 60
max_seconds = 60 * 60
issue_mask = (df['duration'] < min_seconds) | (df['duration'] > max_seconds)

issue_df = df[issue_mask].copy()

if not issue_df.empty:
    issue_df['Comment'] = "Interview duration is outside the expected range (25-45 mins)."
    issue_df['Correction_duration'] = ""
    issue_df['duration_min'] = (issue_df['duration'] / 60).round(2)
    label_dict.update({
        'Comment': "Issue Description",
        'Correction_duration': "Correct Duration (Sec)",
        'duration_min': "Duration in Minutes"
    })
    vars_to_export = variables_to_keep + ['duration', 'duration_min', 'Comment', 'Correction_duration']
    final_issue_data = issue_df[vars_to_export].copy()
    if 'meta' in locals():
        for col in final_issue_data.columns:
            if col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['Duration_Check'] = final_issue_data
    print(f"Added {len(issue_df)} duration errors to Master Log.")
else:
    print("No duration errors found.")

Added 6 duration errors to Master Log.


In [78]:
# A4. How many people live in your household, including yourself?
issue_size = df['A4'] < 2
issue_df = df[issue_size].copy()
if not issue_df.empty:
    issue_df[['Comment', 'Correction_A4']] = [
        "Please confirm if the respondent leaves alone.", ""
            ]
    final_issue_data = issue_df[variables_to_keep + ['A4', 'Comment', 'Correction_A4']].copy()
    if 'meta' in locals():
        for col in final_issue_data.columns:
            if col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['House_size'] = final_issue_data
    print(f"Added {len(issue_df)} logic error to Master Log.")

else:
    print("No logic error found.")

No logic error found.


In [79]:
issue_size = df['A4'] < 2
issue_df = df[issue_size].copy()

if not issue_df.empty:
    issue_df['Comment'] = "Please confirm if the respondent lives alone."
    issue_df['Correction_A4'] = ""
    final_issue_data = issue_df[variables_to_keep + ['A4', 'Comment', 'Correction_A4']].copy()
    if 'meta' in locals():
        for col in final_issue_data.columns:
            if col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['House_size1'] = final_issue_data
    
    print(f"Added {len(issue_df)} logic errors (House Size) to Master Log.")

else:
    print("No house size logic errors found.")

No house size logic errors found.


In [80]:
# A4. How many people live in your household, including yourself? 
# A4a. How many of those earn an income?
issue_mask = df['A4a'] == df['A4']
issue_df = df[issue_mask].copy()
if not issue_df.empty:
    issue_df[['Comment', 'Correction_A4', 'Correction_A4a']] = [
        "Please confirmif everyone in the household is of working age", "", ""
    ]
    final_issue_data = issue_df[variables_to_keep + ['A4', 'A4a', 'Comment', 'Correction_A4', 'Correction_A4a']].copy()
    if 'meta' in locals():
        for col in final_issue_data.columns:
            if col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    output_file = os.path.join(daily_output_path, "Household_Logic.xlsx")
    master_error_log['Issue_Household_Logic'] = final_issue_data
    print(f"Added {len(issue_df)} logic error to Master Log.")

else:
    print("No logic error found.")

Added 7 logic error to Master Log.


In [81]:
# A4. How many people live in your household, including yourself? 
# A4a. How many of those earn an income?
issue_mask = df['A4a'] >= (df['A4'] * 0.7)
issue_df = df[issue_mask].copy()
if not issue_df.empty:
    issue_df[['Comment', 'Correction_A4', 'Correction_A4a']] = [
        "High ratio of earners (>=70% of household). Please confirm if they are of working age", "", ""
    ]
    final_issue_data = issue_df[variables_to_keep + ['A4', 'A4a', 'Comment', 'Correction_A4', 'Correction_A4a']].copy()
    if 'meta' in locals():
        for col in final_issue_data.columns:
            if col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['Issue_Working_Logic'] = final_issue_data
    print(f"Added {len(issue_df)} logic error to Master Log.")
else:
    print("No logic error found.")

Added 20 logic error to Master Log.


In [82]:
# A1.1. What age did you reach on your last birthday? Numeric (Year): 
# A7. How long have you been working in the RMG sector?
Age_exp = df['AGE_HH'] - df['A7_Years']
issue_mask = Age_exp < 14
issue_df = df[issue_mask].copy()

if not issue_df.empty:
    issue_df['Comment'] = "The differnce between age and years of experience is more than 14 year, confirm if the entries are correct ."
    issue_df['Correction_AGE_HH'] = ""
    issue_df['Correction_A7_Years'] = ""
    issue_df['Age_exp'] = Age_exp
    final_issue_data = issue_df[variables_to_keep + ['AGE_HH', 'A7', 'A7_Years', 'Age_exp','Comment', 'Correction_AGE_HH', 'Correction_A7_Years']].copy()
    if 'meta' in locals():
        for col in final_issue_data.columns:
            if col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['Age_Check'] = final_issue_data
    print(f"Added {len(issue_df)} age experience issues to Master Log.")

else:
    print("No age experience issues found.")

Added 4 age experience issues to Master Log.


In [83]:
# A8. How many mobile phone handsets do you own?
# A8b. Is the phone you use most often owned by you or someone else?
issue_mask = (df['A8'] == 0) & (df['A8b'] != 5)
issue_df = df[issue_mask].copy()
if not issue_df.empty:
    issue_df[['Comment', 'Correction_A8', 'Correction_A8b']] = [
        "Respondent has mention they do not have own a phone but shares with husband. What type of phone do they have access to?", "", ""
    ]
    final_issue_data = issue_df[variables_to_keep + ['A8', 'A8b', 'B1_S', 'B2_S', 'Comment', 'Correction_A8', 'Correction_A8b']].copy()
    if 'meta' in locals():
        for col in final_issue_data.columns:
            if col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['phone_usage'] = final_issue_data
    print(f"Added {len(issue_df)} logic error to Master Log.")
else:
    print("No logic error found.")

No logic error found.


#### Section B: Access & Onboarding 

In [84]:
# A8a. What type(s) of mobile phone do you own?: Smartphone?
# B2. Who helped you register for Agami App?
issue_mask = df['A8a_3'] == 0
issue_df = df[issue_mask].copy()
if not issue_df.empty:
    issue_df[['Comment', 'Correction_A8a_1', 'Correction_A8a_2', 'Correction_A8a_3', 'Correction_B2']] = [
        "If the respondent do not own or access a smartphone where did they register for Agami App?", "", "", "", ""
    ]
    final_issue_data = issue_df[variables_to_keep + ['A8a_1', 'A8a_2', 'A8a_3', 'B2', 'B2_S', 'Comment', 'Correction_A8a_1', 'Correction_A8a_2', 'Correction_A8a_3', 'Correction_B2']].copy()
    if 'meta' in locals():
        for col in final_issue_data.columns:
            if col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['phone_access_usage'] = final_issue_data
    print(f"Added {len(issue_df)} logic error to Master Log.")
else:
    print("No logic error found.")

Added 38 logic error to Master Log.


In [85]:
# A8. How many mobile phone handsets do you own?
# B3a. How often do you use the Agami App? 
issue_mask = (df['A8'] == 0) & (df['B3a'] != 7)
issue_df = df[issue_mask].copy()
if not issue_df.empty:
    issue_df[['Comment', 'Correction_A8', 'Correction_A8b', 'Correction_B3a']] = [
        "How are they accessing the app if they do not own a phone", "", "", ""
    ]
    final_issue_data = issue_df[variables_to_keep + ['A8', 'A8b', 'B3a', 'Comment', 'Correction_A8', 'Correction_A8b', 'Correction_B3a']].copy()
    if 'meta' in locals():
        for col in final_issue_data.columns:
            if col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['app_usage'] = final_issue_data
    print(f"Added {len(issue_df)} logic error to Master Log.")
else:
    print("No logic error found.")

No logic error found.


In [86]:
# A8a. What type(s) of mobile phone do you own?: Smartphone?
# B4. Do you log in to Agami App by yourself or with someone’s help?
issue_mask = (df['A8a_3'] == 0) & ((df['B4'] == 1) | (df['B4'] == 2)) 
issue_df = df[issue_mask].copy()
if not issue_df.empty:
    issue_df[['Comment', 'Correction_A8a_3', 'CorrectionB4']] = [
        "Where do the respondent get the phone to log in the app?", "", ""
    ]
    final_issue_data = issue_df[variables_to_keep + ['A8a_1', 'A8a_2', 'A8a_3', 'B4', 'Comment', 'Correction_A8a_3', 'CorrectionB4']].copy()
    if 'meta' in locals():
        for col in final_issue_data.columns:
            if col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['app_access'] = final_issue_data
    print(f"Added {len(issue_df)} logic error to Master Log.")
else:
    print("No logic error found.")

Added 5 logic error to Master Log.


#### Section C: Usage & Product Uptake 

In [87]:
# B3a. How often do you use the Agami App? 
# C2c. How often are you using the feature in the Agami App?

cond_daily = (df['C2c_1'] == 1) & (df['B3a'] > 1)
cond_weekly = (df['C2c_1'] == 2) & (df['B3a'] > 3)
cond_monthly = (df['C2c_1'] == 3) & (df['B3a'] > 5)
issue_mask = cond_daily | cond_weekly | cond_monthly
issue_df = df[issue_mask].copy()

if not issue_df.empty:
    issue_df['Comment'] = "Frequency mismatch: Specific action is more frequent than general app usage."
    issue_df['Correction_B3a'] = ""
    issue_df['Correction_C2c_1'] = ""
    final_issue_data = issue_df[variables_to_keep + ['B3a', 'C2c_1', 'Comment', 'Correction_B3a', 'Correction_C2c_1']].copy()
    if 'meta' in locals():
        for col in ['B3a', 'C2c_1']:
            if col in final_issue_data.columns and col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['C2c_1'] = final_issue_data
    print(f"Added {len(issue_df)} frequency mismatch errors to Master Log.")
else:
    print("No frequency mismatch errors found.")

Added 4 frequency mismatch errors to Master Log.


In [88]:
# B3a. How often do you use the Agami App? 
# C2c. How often are you using the feature in the Agami App?

cond_daily = (df['C2c_2'] == 1) & (df['B3a'] > 1)
cond_weekly = (df['C2c_2'] == 2) & (df['B3a'] > 3)
cond_monthly = (df['C2c_2'] == 3) & (df['B3a'] > 5)
issue_mask = cond_daily | cond_weekly | cond_monthly
issue_df = df[issue_mask].copy()

if not issue_df.empty:
    issue_df['Comment'] = "Frequency mismatch: Specific action is more frequent than general app usage."
    issue_df['Correction_B3a'] = ""
    issue_df['Correction_C2c_2'] = ""
    final_issue_data = issue_df[variables_to_keep + ['B3a', 'C2c_2', 'Comment', 'Correction_B3a', 'Correction_C2c_2']].copy()
    if 'meta' in locals():
        for col in ['B3a', 'C2c_2']:
            if col in final_issue_data.columns and col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['C2c_2'] = final_issue_data
    print(f"Added {len(issue_df)} frequency mismatch errors to Master Log.")
else:
    print("No frequency mismatch errors found.")

Added 1 frequency mismatch errors to Master Log.


In [89]:
# B3a. How often do you use the Agami App? 
# C2c. How often are you using the feature in the Agami App?

cond_daily = (df['C2c_3'] == 1) & (df['B3a'] > 1)
cond_weekly = (df['C2c_3'] == 2) & (df['B3a'] > 3)
cond_monthly = (df['C2c_3'] == 3) & (df['B3a'] > 5)
issue_mask = cond_daily | cond_weekly | cond_monthly
issue_df = df[issue_mask].copy()

if not issue_df.empty:
    issue_df['Comment'] = "Frequency mismatch: Specific action is more frequent than general app usage."
    issue_df['Correction_B3a'] = ""
    issue_df['Correction_C2c_3'] = ""
    final_issue_data = issue_df[variables_to_keep + ['B3a', 'C2c_3', 'Comment', 'Correction_B3a', 'Correction_C2c_3']].copy()
    if 'meta' in locals():
        for col in ['B3a', 'C2c_3']:
            if col in final_issue_data.columns and col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['C2c_3'] = final_issue_data
    print(f"Added {len(issue_df)} frequency mismatch errors to Master Log.")
else:
    print("No frequency mismatch errors found.")

Added 2 frequency mismatch errors to Master Log.


In [90]:
# B3a. How often do you use the Agami App? 
# C2c. How often are you using the feature in the Agami App?

cond_daily = (df['C2c_4'] == 1) & (df['B3a'] > 1)
cond_weekly = (df['C2c_4'] == 2) & (df['B3a'] > 3)
cond_monthly = (df['C2c_4'] == 3) & (df['B3a'] > 5)
issue_mask = cond_daily | cond_weekly | cond_monthly
issue_df = df[issue_mask].copy()

if not issue_df.empty:
    issue_df['Comment'] = "Frequency mismatch: Specific action is more frequent than general app usage."
    issue_df['Correction_B3a'] = ""
    issue_df['Correction_C2c_4'] = ""
    final_issue_data = issue_df[variables_to_keep + ['B3a', 'C2c_4', 'Comment', 'Correction_B3a', 'Correction_C2c_4']].copy()
    if 'meta' in locals():
        for col in ['B3a', 'C2c_4']:
            if col in final_issue_data.columns and col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['C2c_4'] = final_issue_data
    print(f"Added {len(issue_df)} frequency mismatch errors to Master Log.")
else:
    print("No frequency mismatch errors found.")

Added 3 frequency mismatch errors to Master Log.


In [91]:
# B3a. How often do you use the Agami App? 
# C2c. How often are you using the feature in the Agami App?

cond_daily = (df['C2c_5'] == 1) & (df['B3a'] > 1)
cond_weekly = (df['C2c_5'] == 2) & (df['B3a'] > 3)
cond_monthly = (df['C2c_5'] == 3) & (df['B3a'] > 5)
issue_mask = cond_daily | cond_weekly | cond_monthly
issue_df = df[issue_mask].copy()

if not issue_df.empty:
    issue_df['Comment'] = "Frequency mismatch: Specific action is more frequent than general app usage."
    issue_df['Correction_B3a'] = ""
    issue_df['Correction_C2c_5'] = ""
    final_issue_data = issue_df[variables_to_keep + ['B3a', 'C2c_5', 'Comment', 'Correction_B3a', 'Correction_C2c_5']].copy()
    if 'meta' in locals():
        for col in ['B3a', 'C2c_5']:
            if col in final_issue_data.columns and col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['C2c_5'] = final_issue_data
    print(f"Added {len(issue_df)} frequency mismatch errors to Master Log.")
else:
    print("No frequency mismatch errors found.")

Added 1 frequency mismatch errors to Master Log.


In [92]:
# B3a. How often do you use the Agami App? 
# C2c. How often are you using the feature in the Agami App?

cond_daily = (df['C2c_6'] == 1) & (df['B3a'] > 1)
cond_weekly = (df['C2c_6'] == 2) & (df['B3a'] > 3)
cond_monthly = (df['C2c_6'] == 3) & (df['B3a'] > 5)
issue_mask = cond_daily | cond_weekly | cond_monthly
issue_df = df[issue_mask].copy()

if not issue_df.empty:
    issue_df['Comment'] = "Frequency mismatch: Specific action is more frequent than general app usage."
    issue_df['Correction_B3a'] = ""
    issue_df['Correction_C2c_6'] = ""
    final_issue_data = issue_df[variables_to_keep + ['B3a', 'C2c_6', 'Comment', 'Correction_B3a', 'Correction_C2c_6']].copy()
    if 'meta' in locals():
        for col in ['B3a', 'C2c_6']:
            if col in final_issue_data.columns and col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['C2c_6'] = final_issue_data
    print(f"Added {len(issue_df)} frequency mismatch errors to Master Log.")
else:
    print("No frequency mismatch errors found.")

No frequency mismatch errors found.


In [93]:
# B3a. How often do you use the Agami App? 
# C2c. How often are you using the feature in the Agami App?

cond_daily = (df['C2c_7'] == 1) & (df['B3a'] > 1)
cond_weekly = (df['C2c_7'] == 2) & (df['B3a'] > 3)
cond_monthly = (df['C2c_7'] == 3) & (df['B3a'] > 5)
issue_mask = cond_daily | cond_weekly | cond_monthly
issue_df = df[issue_mask].copy()

if not issue_df.empty:
    issue_df['Comment'] = "Frequency mismatch: Specific action is more frequent than general app usage."
    issue_df['Correction_B3a'] = ""
    issue_df['Correction_C2c_7'] = ""
    final_issue_data = issue_df[variables_to_keep + ['B3a', 'C2c_7', 'Comment', 'Correction_B3a', 'Correction_C2c_7']].copy()
    if 'meta' in locals():
        for col in ['B3a', 'C2c_7']:
            if col in final_issue_data.columns and col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['C2c_7'] = final_issue_data
    print(f"Added {len(issue_df)} frequency mismatch errors to Master Log.")
else:
    print("No frequency mismatch errors found.")

No frequency mismatch errors found.


In [94]:
# B3a. How often do you use the Agami App? 
# C2c. How often are you using the feature in the Agami App?

cond_daily = (df['C2c_8'] == 1) & (df['B3a'] > 1)
cond_weekly = (df['C2c_8'] == 2) & (df['B3a'] > 3)
cond_monthly = (df['C2c_8'] == 3) & (df['B3a'] > 5)
issue_mask = cond_daily | cond_weekly | cond_monthly
issue_df = df[issue_mask].copy()

if not issue_df.empty:
    issue_df['Comment'] = "Frequency mismatch: Specific action is more frequent than general app usage."
    issue_df['Correction_B3a'] = ""
    issue_df['Correction_C2c_8'] = ""
    final_issue_data = issue_df[variables_to_keep + ['B3a', 'C2c_8', 'Comment', 'Correction_B3a', 'Correction_C2c_8']].copy()
    if 'meta' in locals():
        for col in ['B3a', 'C2c_8']:
            if col in final_issue_data.columns and col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['C2c_8'] = final_issue_data
    print(f"Added {len(issue_df)} frequency mismatch errors to Master Log.")
else:
    print("No frequency mismatch errors found.")

No frequency mismatch errors found.


In [95]:
# B3a. How often do you use the Agami App? 
# C2c. How often are you using the feature in the Agami App?

cond_daily = (df['C2c_9'] == 1) & (df['B3a'] > 1)
cond_weekly = (df['C2c_9'] == 2) & (df['B3a'] > 3)
cond_monthly = (df['C2c_9'] == 3) & (df['B3a'] > 5)
issue_mask = cond_daily | cond_weekly | cond_monthly
issue_df = df[issue_mask].copy()

if not issue_df.empty:
    issue_df['Comment'] = "Frequency mismatch: Specific action is more frequent than general app usage."
    issue_df['Correction_B3a'] = ""
    issue_df['Correction_C2c_9'] = ""
    final_issue_data = issue_df[variables_to_keep + ['B3a', 'C2c_9', 'Comment', 'Correction_B3a', 'Correction_C2c_9']].copy()
    if 'meta' in locals():
        for col in ['B3a', 'C2c_9']:
            if col in final_issue_data.columns and col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['C2c_9'] = final_issue_data
    print(f"Added {len(issue_df)} frequency mismatch errors to Master Log.")
else:
    print("No frequency mismatch errors found.")

No frequency mismatch errors found.


In [96]:
# B3a. How often do you use the Agami App? 
# C2c. How often are you using the feature in the Agami App?

cond_daily = (df['C2c_10'] == 1) & (df['B3a'] > 1)
cond_weekly = (df['C2c_10'] == 2) & (df['B3a'] > 3)
cond_monthly = (df['C2c_10'] == 3) & (df['B3a'] > 5)
issue_mask = cond_daily | cond_weekly | cond_monthly
issue_df = df[issue_mask].copy()

if not issue_df.empty:
    issue_df['Comment'] = "Frequency mismatch: Specific action is more frequent than general app usage."
    issue_df['Correction_B3a'] = ""
    issue_df['Correction_C2c_10'] = ""
    final_issue_data = issue_df[variables_to_keep + ['B3a', 'C2c_10', 'Comment', 'Correction_B3a', 'Correction_C2c_10']].copy()
    if 'meta' in locals():
        for col in ['B3a', 'C2c_10']:
            if col in final_issue_data.columns and col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['C2c_10'] = final_issue_data
    print(f"Added {len(issue_df)} frequency mismatch errors to Master Log.")
else:
    print("No frequency mismatch errors found.")

No frequency mismatch errors found.


In [97]:
# B3a. How often do you use the Agami App? 
# C2c. How often are you using the feature in the Agami App?

cond_daily = (df['C2c_11'] == 1) & (df['B3a'] > 1)
cond_weekly = (df['C2c_11'] == 2) & (df['B3a'] > 3)
cond_monthly = (df['C2c_11'] == 3) & (df['B3a'] > 5)
issue_mask = cond_daily | cond_weekly | cond_monthly
issue_df = df[issue_mask].copy()

if not issue_df.empty:
    issue_df['Comment'] = "Frequency mismatch: Specific action is more frequent than general app usage."
    issue_df['Correction_B3a'] = ""
    issue_df['Correction_C2c_11'] = ""
    final_issue_data = issue_df[variables_to_keep + ['B3a', 'C2c_11', 'Comment', 'Correction_B3a', 'Correction_C2c_11']].copy()
    if 'meta' in locals():
        for col in ['B3a', 'C2c_11']:
            if col in final_issue_data.columns and col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['C2c_11'] = final_issue_data
    print(f"Added {len(issue_df)} frequency mismatch errors to Master Log.")
else:
    print("No frequency mismatch errors found.")

No frequency mismatch errors found.


In [98]:
# B3a. How often do you use the Agami App? 
# C2c. How often are you using the feature in the Agami App?

cond_daily = (df['C2c_12'] == 1) & (df['B3a'] > 1)
cond_weekly = (df['C2c_12'] == 2) & (df['B3a'] > 3)
cond_monthly = (df['C2c_12'] == 3) & (df['B3a'] > 5)
issue_mask = cond_daily | cond_weekly | cond_monthly
issue_df = df[issue_mask].copy()

if not issue_df.empty:
    issue_df['Comment'] = "Frequency mismatch: Specific action is more frequent than general app usage."
    issue_df['Correction_B3a'] = ""
    issue_df['Correction_C2c_12'] = ""
    final_issue_data = issue_df[variables_to_keep + ['B3a', 'C2c_12', 'Comment', 'Correction_B3a', 'Correction_C2c_12']].copy()
    if 'meta' in locals():
        for col in ['B3a', 'C2c_12']:
            if col in final_issue_data.columns and col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['C2c_12'] = final_issue_data
    print(f"Added {len(issue_df)} frequency mismatch errors to Master Log.")
else:
    print("No frequency mismatch errors found.")

No frequency mismatch errors found.


In [99]:
# B3a. How often do you use the Agami App? 
# C2c. How often are you using the feature in the Agami App?

cond_daily = (df['C2c_13'] == 1) & (df['B3a'] > 1)
cond_weekly = (df['C2c_13'] == 2) & (df['B3a'] > 3)
cond_monthly = (df['C2c_13'] == 3) & (df['B3a'] > 5)
issue_mask = cond_daily | cond_weekly | cond_monthly
issue_df = df[issue_mask].copy()

if not issue_df.empty:
    issue_df['Comment'] = "Frequency mismatch: Specific action is more frequent than general app usage."
    issue_df['Correction_B3a'] = ""
    issue_df['Correction_C2c_13'] = ""
    final_issue_data = issue_df[variables_to_keep + ['B3a', 'C2c_13', 'Comment', 'Correction_B3a', 'Correction_C2c_13']].copy()
    if 'meta' in locals():
        for col in ['B3a', 'C2c_13']:
            if col in final_issue_data.columns and col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['Frequency_Logic'] = final_issue_data
    print(f"Added {len(issue_df)} frequency mismatch errors to Master Log.")
else:
    print("No frequency mismatch errors found.")

No frequency mismatch errors found.


#### Section D: Savings & Loan Behaviour 

In [100]:
# D4. Approximately how much do you save each month through?
issue_mask = df['D4._1'] > 5000
issue_df = df[issue_mask].copy()
if not issue_df.empty:
    issue_df[['Comment', 'Correction_D4._1']] = [
        "High savings amount detected. Please confirm the entry.", ""
    ]
    final_issue_data = issue_df[variables_to_keep + ['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D3_1', 'D4._1', 'D6_1', 'Comment', 'Correction_D4._1']].copy()
    if 'meta' in locals():
        for col in final_issue_data.columns:
            if col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['amount_issue_1'] = final_issue_data
    print(f"Added {len(issue_df)} logic error to Master Log.")
else:
    print("No logic error found.")

Added 2 logic error to Master Log.


In [101]:
# D4. Approximately how much do you save each month through?
issue_mask = df['D4._2'] > 5000
issue_df = df[issue_mask].copy()
if not issue_df.empty:
    issue_df[['Comment', 'Correction_D4._2']] = [
        "High savings amount detected. Please confirm the entry.", ""
    ]
    final_issue_data = issue_df[variables_to_keep + ['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D3_2', 'D4._2', 'D6_2', 'Comment', 'Correction_D4._2']].copy()
    if 'meta' in locals():
        for col in final_issue_data.columns:
            if col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['amount_issue_2'] = final_issue_data
    print(f"Added {len(issue_df)} logic error to Master Log.")
else:
    print("No logic error found.")

Added 2 logic error to Master Log.


In [102]:
# D4. Approximately how much do you save each month through?
issue_mask = df['D4._3'] > 5000
issue_df = df[issue_mask].copy()
if not issue_df.empty:
    issue_df[['Comment', 'Correction_D4._3']] = [
        "High savings amount detected. Please confirm the entry.", ""
    ]
    final_issue_data = issue_df[variables_to_keep + ['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D3_3', 'D4._3', 'D6_3', 'Comment', 'Correction_D4._3']].copy()
    if 'meta' in locals():
        for col in final_issue_data.columns:
            if col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['amount_issue_3'] = final_issue_data
    print(f"Added {len(issue_df)} logic error to Master Log.")
else:
    print("No logic error found.")

Added 1 logic error to Master Log.


In [103]:
# D4. Approximately how much do you save each month through?
issue_mask = df['D4._4'] > 5000
issue_df = df[issue_mask].copy()
if not issue_df.empty:
    issue_df[['Comment', 'Correction_D4._4']] = [
        "High savings amount detected. Please confirm the entry.", ""
    ]
    final_issue_data = issue_df[variables_to_keep + ['D1_1', 'D1_2', 'D1_3', 'D1_4', 'D1_5', 'D1_6', 'D1_7', 'D1_8', 'D3_4', 'D4._4', 'D6_4', 'Comment', 'Correction_D4._4']].copy()
    if 'meta' in locals():
        for col in final_issue_data.columns:
            if col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['amount_issue_2'] = final_issue_data
    print(f"Added {len(issue_df)} logic error to Master Log.")
else:
    print("No logic error found.")

No logic error found.


In [104]:
# D7 .Where do you currently borrow from?
issue_mask = df['D10_1'] > 100000
issue_df = df[issue_mask].copy()
if not issue_df.empty:
    issue_df[['Comment', 'Correction_D10_1']] = [
        "High borrowed amount detected. Please confirm the entry.", ""
    ]
    final_issue_data = issue_df[variables_to_keep + ['D7_1', 'D7_2', 'D7_3', 'D7_4', 'D7_5', 'D7_6', 'D7_7', 'D7_8', 'D9_1', 'D10_1', 'D12_1', 'Comment', 'Correction_D10_1']].copy()
    if 'meta' in locals():
        for col in final_issue_data.columns:
            if col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['loan_issue_1'] = final_issue_data
    print(f"Added {len(issue_df)} logic error to Master Log.")
else:
    print("No logic error found.")

Added 7 logic error to Master Log.


In [105]:
# D7 .Where do you currently borrow from?
issue_mask = df['D10_2'] > 100000
issue_df = df[issue_mask].copy()
if not issue_df.empty:
    issue_df[['Comment', 'Correction_D10_2']] = [
        "High borrowed amount detected. Please confirm the entry.", ""
    ]
    final_issue_data = issue_df[variables_to_keep + ['D7_1', 'D7_2', 'D7_3', 'D7_4', 'D7_5', 'D7_6', 'D7_7', 'D7_8', 'D9_2', 'D10_2', 'D12_2', 'Comment', 'Correction_D10_2']].copy()
    if 'meta' in locals():
        for col in final_issue_data.columns:
            if col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['loan_issue_2'] = final_issue_data
    print(f"Added {len(issue_df)} logic error to Master Log.")
else:
    print("No logic error found.")

No logic error found.


In [106]:
# D7 .Where do you currently borrow from?
issue_mask = df['D10_3'] > 100000
issue_df = df[issue_mask].copy()
if not issue_df.empty:
    issue_df[['Comment', 'Correction_D10_3']] = [
        "High borrowed amount detected. Please confirm the entry.", ""
    ]
    final_issue_data = issue_df[variables_to_keep + ['D7_1', 'D7_2', 'D7_3', 'D7_4', 'D7_5', 'D7_6', 'D7_7', 'D7_8', 'D9_3', 'D10_3', 'D12_3', 'Comment', 'Correction_D10_3']].copy()
    if 'meta' in locals():
        for col in final_issue_data.columns:
            if col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['loan_issue_3'] = final_issue_data
    print(f"Added {len(issue_df)} logic error to Master Log.")
else:
    print("No logic error found.")

No logic error found.


In [107]:
# D7 .Where do you currently borrow from?
issue_mask = df['D10_4'] > 100000
issue_df = df[issue_mask].copy()
if not issue_df.empty:
    issue_df[['Comment', 'Correction_D10_4']] = [
        "High borrowed amount detected. Please confirm the entry.", ""
    ]
    final_issue_data = issue_df[variables_to_keep + ['D7_1', 'D7_2', 'D7_3', 'D7_4', 'D7_5', 'D7_6', 'D7_7', 'D7_8', 'D9_4', 'D10_4', 'D12_4', 'Comment', 'Correction_D10_4']].copy()
    if 'meta' in locals():
        for col in final_issue_data.columns:
            if col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])
    master_error_log['loan_issue_4'] = final_issue_data
    print(f"Added {len(issue_df)} logic error to Master Log.")
else:
    print("No logic error found.")

No logic error found.


#### Section G. Financial Control & Decision-Making 

In [108]:
# Missing values for G2 and G2a
# load the SPSS file
file_name = "BRAC - BASELINE EVALUATION_WIDE V006.sav"

# Load data AND metadata
df, meta = pyreadstat.read_sav(file_name, apply_value_formats=False)
label_dict = meta.column_names_to_labels

issue_mask = (df['B3a'] != 7) & (df['G2'].isna()) & (df['G2a'].isna())

issue_df = df[issue_mask].copy()

if not issue_df.empty:
    issue_df['Comment'] = "G2 and G2a are missing even though B3a indicates app usage (Not 7)."
    issue_df['Correction_B3a'] = ""
    issue_df['Correction_G2'] = ""
    issue_df['Correction_G2a'] = ""

    vars_to_export = variables_to_keep + ['B3a', 'G2', 'G2a', 'Comment', 'Correction_B3a', 'Correction_G2', 'Correction_G2a']
    final_issue_data = issue_df[vars_to_export].copy()

    if 'meta' in locals():
        for col in ['B3a']:
            if col in final_issue_data.columns and col in meta.variable_value_labels:
                final_issue_data[col] = final_issue_data[col].map(meta.variable_value_labels[col]).fillna(final_issue_data[col])

    master_error_log['Missing_G2_G2a'] = final_issue_data
    
    print(f"Added {len(issue_df)} missing G2/G2a errors to Master Log.")

else:
    print("No missing G2/G2a errors found.")

Added 12 missing G2/G2a errors to Master Log.


#### Adding color codes

In [109]:
def apply_worksheet_styling(workbook, worksheet, df, keep_vars):
    base_fmt = workbook.add_format({'valign': 'top'})
    correction_data_fmt = workbook.add_format({'bg_color': '#92D050', 'valign': 'top'})
    comment_data_fmt = workbook.add_format({'bg_color': '#FFC000', 'valign': 'top', 'text_wrap': True})
    issue_data_fmt = workbook.add_format({
        'bg_color': '#88194B', 
        'font_color': '#FFFFFF', 
        'valign': 'top'
    })
    start_row = 2
    end_row = start_row + len(df) - 1
    
    if len(df) == 0:
        return
    for col_num, col_name in enumerate(df.columns):
        col_width = 15
        
        if col_name == 'Comment':
            col_width = 40
        elif col_name.startswith('Correction'):
            col_width = 20
        elif col_name in keep_vars:
            col_width = 15
        elif col_name == 'A4':
            col_width = 10
            
        worksheet.set_column(col_num, col_num, col_width, base_fmt)
        if col_name.startswith('Correction'):
            worksheet.conditional_format(start_row, col_num, end_row, col_num, {
                'type': 'no_blanks', 'format': correction_data_fmt
            })
        elif col_name == 'Comment':
            worksheet.conditional_format(start_row, col_num, end_row, col_num, {
                'type': 'no_blanks', 'format': comment_data_fmt
            })
            
        elif col_name in keep_vars:
            pass
        else:
            worksheet.conditional_format(start_row, col_num, end_row, col_num, {
                'type': 'no_blanks', 'format': issue_data_fmt
            })

#### Merging the error logs

In [110]:
# def save_master_workbook(error_collection, label_dict, folder_path, filename=f"BRAC RMG Workers Impact Evaluation issues {today_str}.xlsx"):

def save_master_workbook(error_collection, label_dict, folder_path, keep_vars, filename=f"BRAC RMG Workers Impact Evaluation issues {today_str}.xlsx"):
    if not error_collection:
        print("No errors found today! No file generated.")
        return

    full_path = os.path.join(folder_path, filename)
    with pd.ExcelWriter(full_path, engine='xlsxwriter') as writer:
        workbook = writer.book
        std_header_fmt = workbook.add_format({
            'bold': True, 'text_wrap': True, 'valign': 'top'
        })
        corr_header_fmt = workbook.add_format({
            'bold': True, 'text_wrap': True, 'valign': 'top', 
            'bg_color': '#92D050'
        })
        var_name_fmt = workbook.add_format({
            'bold': True, 'align': 'center'
        })
        corr_var_fmt = workbook.add_format({
            'bold': True, 'align': 'center',
            'bg_color': '#92D050'
        })

        for sheet_name, data in error_collection.items():
            # Write Data
            data.to_excel(writer, sheet_name=sheet_name, startrow=2, header=False, index=False)
            worksheet = writer.sheets[sheet_name]
            
            # Write Headers
            header_labels = [label_dict.get(col, col) for col in data.columns]
            variable_names = data.columns.tolist()
            
            for col_num, col_name in enumerate(variable_names):
                if col_name.startswith('Correction'):
                    top_fmt = corr_header_fmt
                    btm_fmt = corr_var_fmt
                else:
                    top_fmt = std_header_fmt
                    btm_fmt = var_name_fmt
                
                worksheet.write(0, col_num, header_labels[col_num], top_fmt)
                worksheet.write(1, col_num, col_name, btm_fmt)
            
            apply_worksheet_styling(workbook, worksheet, data, keep_vars)
                
    print(f"SUCCESS: Master workbook saved at {full_path}")

save_master_workbook(master_error_log, label_dict, daily_output_path, variables_to_keep)

SUCCESS: Master workbook saved at D:/OneDrive/Dalberg Research Projects/BRAC/Error Logs\2025-12-22\BRAC RMG Workers Impact Evaluation issues 2025-12-22.xlsx


### Extracting open ends

In [111]:
# Variables that requires translations
file_name = "BRAC - BASELINE EVALUATION_WIDE V004.sav"

# Load data AND metadata
df, meta = pyreadstat.read_sav(file_name, apply_value_formats=False)
label_dict = meta.column_names_to_labels

var_to_keep_for_translation = ['A3_S', 'A6_S', 'B1_S', 'B2_S', 'B4a_S', 'B6_S', 'B8a_S', 'B9a_S', 'B10a_S', 'B10c_1', 'B10c_S_1', 'B10c_2', 
                               'B10c_S_2', 'B10c_3', 'B10c_S_3', 'C2', 'C2a', 'C2a_S', 'C2b', 'C2d_1', 'C2d_S_1', 'C2d_S_2', 'C2d_S_3', 'C2d_S_4', 
                               'C2d_S_5', 'C2d_S_6', 'C2d_S_7', 'C2d_S_8', 'C2d_S_9', 'C2d_S_10', 'C2d_S_11', 'C2d_S_12', 'C2d_S_13',
                               'C3_S', 'C4a_S', 'C5a_S', 'C6a_S', 'C10a_S', 'C11_S', 'C12_S', 'C13_S', 'D1_S', 'D2_S', 'D5_S_1', 'D6_S_1', 'D5_S_2', 'D6_S_2', 'D5_S_3', 'D6_S_3', 
                               'D5_S_4', 'D6_S_4', 'D7_S', 'D8_S', 'D11_S_1', 'D12_S_1', 'D11_S_2', 'D12_S_2', 'D11_S_3', 'D12_S_3', 'D11_S_4', 'D12_S_4', 'D13a_S', 'D14a_S',
                               'E1_S', 'E2a_S', 'E4', 'F1a_S', 'F1b_S', 'F2', 'F5', 'F5_S', 'G1_S', 'G1a_S', 'G1b_S', 'G2_S', 'G2a_S', 'G4a_S', 'G5a_S', 'G6a_S'
                               ]

translation_bucket = {}
for var in var_to_keep_for_translation:
    if var in df.columns:
        subset = df[['KEY', var]].dropna(subset=[var])
        subset = subset[subset[var] != ''] 
        subset = subset.sort_values(by=var, ascending=False)
        
        if not subset.empty:
            subset['Translation'] = ""
            sheet_name = var[:31] 
            translation_bucket[sheet_name] = subset
            
    else:
        print(f"Warning: Variable {var} not found in dataset.")

print(f"Prepared {len(translation_bucket)} sheets for translation.")

Prepared 60 sheets for translation.


In [112]:
def save_translation_workbook(data_dict, label_dict, folder_path, filename="Translations.xlsx"):
    if not data_dict:
        print("No data to translate.")
        return

    full_path = os.path.join(folder_path, filename)

    with pd.ExcelWriter(full_path, engine='xlsxwriter') as writer:
        workbook = writer.book
        static_header_fmt = workbook.add_format({
            'bold': True, 'text_wrap': True, 'valign': 'top'
        })
    
        input_header_fmt = workbook.add_format({
            'bold': True, 'text_wrap': True, 'valign': 'top' 
                   })

        text_fmt = workbook.add_format({'text_wrap': True, 'valign': 'top', 'border': 1})
        input_fmt = workbook.add_format({'text_wrap': True, 'valign': 'top', 'border': 1, 'bg_color': '#EBF1DE'}) # Light Green

        for sheet_name, data in data_dict.items():
            data.to_excel(writer, sheet_name=sheet_name, startrow=2, header=False, index=False)
            worksheet = writer.sheets[sheet_name]
            id_var_name = data.columns[0]
            original_var_name = data.columns[1]
            id_label = label_dict.get(id_var_name, id_var_name) 
            question_label = label_dict.get(original_var_name, "No Label Found")
            worksheet.write(0, 0, id_label, static_header_fmt)
            worksheet.write(0, 1, question_label, static_header_fmt)
            worksheet.write(0, 2, "Please enter English translation here", input_header_fmt)
            worksheet.write(1, 0, id_var_name, static_header_fmt)
            worksheet.write(1, 1, original_var_name, static_header_fmt)
            worksheet.write(1, 2, "Translation", input_header_fmt)
            worksheet.set_column(0, 0, 15)
            worksheet.set_column(1, 1, 50)
            worksheet.set_column(2, 2, 50)

    print(f"SUCCESS: Translation workbook saved at {full_path}")

# Define the main path
Working_path = "D:/OneDrive/Dalberg Research Projects/BRAC"
translation_output_path = os.path.join(Working_path, "Translations")
if not os.path.exists(translation_output_path):
    os.makedirs(translation_output_path)
    print(f"Created new folder: {translation_output_path}")
else:
    print(f"Folder already exists: {translation_output_path}")
today_str = datetime.now().strftime("%Y-%m-%d")
translation_filename = f"BRAC_Translation_Request_{today_str}.xlsx"

save_translation_workbook(
    translation_bucket, 
    label_dict, 
    translation_output_path, 
    filename=translation_filename
)

Folder already exists: D:/OneDrive/Dalberg Research Projects/BRAC\Translations
SUCCESS: Translation workbook saved at D:/OneDrive/Dalberg Research Projects/BRAC\Translations\BRAC_Translation_Request_2025-12-22.xlsx
